# DeepSafety Explorer

This notebook gives a quick tour of the current DeepSafety platform:

- inspect the registered models
- review equations and constants
- run calculations directly from the Python library
- generate screening impact zones
- exercise the API locally with FastAPI's `TestClient`


In [ ]:
from pprint import pprint

from fastapi.testclient import TestClient

from deepsafety.api import create_app
from deepsafety.catalog import MODEL_REGISTRY, run_model
from deepsafety.constants import list_constants


In [ ]:
sorted(MODEL_REGISTRY.keys())

## Constants

The registry below shows the current default constants exposed through the API.


In [ ]:
pprint(list_constants())

## Library-Level Calculations

Run a flammability calculation directly through the model registry.


In [ ]:
flammability_outputs, flammability_constants = run_model(
    "fire.flammability_limits",
    {"temp_c": 50, "lfl_20c": 2.1, "ufl_20c": 9.5},
)
pprint(flammability_outputs)
pprint(flammability_constants)

Generate a thermal impact radius for a selected fire threshold.


In [ ]:
fire_zone_outputs, fire_zone_constants = run_model(
    "fire.point_source_heat_flux_radius",
    {
        "burning_rate_kg_s": 4.5,
        "heat_of_combustion_kj_kg": 46000,
        "impact_threshold_kw_m2": 12.5,
    },
)
pprint(fire_zone_outputs)
pprint(fire_zone_constants)

Generate a leak screening radius using released mass and a concentration threshold.


In [ ]:
leak_zone_outputs, leak_zone_constants = run_model(
    "dispersion.gaussian_puff_screening_radius",
    {
        "released_mass_kg": 72.0,
        "concentration_threshold_kg_m3": 0.02,
        "stability_class": "D",
    },
)
pprint(leak_zone_outputs)
pprint(leak_zone_constants)

## API-Level Exploration

Use FastAPI's test client for local exploration without starting a server.


In [ ]:
client = TestClient(create_app())

models_response = client.get("/models")
models_response.status_code, models_response.json()[:3]

In [ ]:
impact_zone_response = client.post(
    "/gis/impact-zones",
    json={
        "scenario_type": "leak",
        "source": {
            "latitude": 51.5074,
            "longitude": -0.1278,
            "label": "Gas Line Segment",
        },
        "asset": {
            "line_pressure_kpa": 6000,
            "mass_flow_kg_s": 1.2,
            "gas_temperature_c": 18,
            "leak_duration_s": 60,
            "stability_class": "D",
        },
        "criteria": [
            {
                "label": "Concern threshold",
                "threshold": 0.02,
                "unit": "kg/m^3",
            }
        ],
    },
)
impact_zone_response.status_code

In [ ]:
impact_zone_payload = impact_zone_response.json()
pprint(impact_zone_payload["zones"])
impact_zone_payload["geojson"]["features"][1]["geometry"]["type"]